<a href="https://colab.research.google.com/github/lpatamia1/lpatamia1/blob/main/MSPB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medicare Hospital Spending Analysis (State-Level)

This project analyzes **Medicare Hospital Spending Per Beneficiary (MSPB)** data to identify states with unusually high or low spending compared to predicted benchmarks.  
The workflow demonstrates **data cleaning, feature engineering, regression modeling, and exporting insights** for visualization.

---

## 📌 Workflow Overview

1. **Import Libraries**  
   - pandas, numpy → data handling  
   - scikit-learn → regression + metrics  

2. **Load and Clean Data**  
   - Import CSV (`Medicare_Hospital_Spending_Per_Patient-State.csv`)  
   - Standardize column names to snake_case  
   - Ensure target column (`score`) is numeric  

3. **Feature Engineering**  
   - Encode states into numeric IDs (`state_id`)  
   - Define target (`score`) and feature (`state_id`)  

4. **Model Training**  
   - Train a simple Linear Regression model  
   - Predict expected MSPB scores per state  

5. **Error Analysis**  
   - Calculate prediction error (`actual - predicted`)  
   - Positive error → higher-than-expected spending (possible inefficiency)  
   - Negative error → lower-than-expected spending (efficiency or under-utilization)  

6. **Model Evaluation**  
   - Compute R² score  
   - Identify top 5 outlier states  

7. **Visualization (Plotly)**
- Choropleth Map: Geographic hotspots of overspending/underspending.
- Bar Chart: Top 10 states for AMA audits.
- Scatter Plot: Technical validation of the methodology.

---

## ✅ Deliverables
- Cleaned and analyzed dataset  
- Outlier states flagged by prediction error  
- Model performance summary (R², outlier report)  
- Visualization

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load the data
file_path = 'Medicare_Hospital_Spending_Per_Patient-State.csv'
df = pd.read_csv(file_path)

# 2. Clean and standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('[^a-zA-Z0-9_]', '', regex=True)

# 3. Identify and clean the Target Variable
# The 'Score' is typically the index or ratio (e.g., 1.00 = National Average)
# Assuming 'score' is the core metric for Spending Per Beneficiary
TARGET_COLUMN = 'score'
STATE_COLUMN = 'state'

# Ensure the target column is numeric (convert errors to NaN and drop)
df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors='coerce')
df = df.dropna(subset=[TARGET_COLUMN]).copy()

# 4. Feature Engineering: Create a numerical feature for 'State'
# The prediction error will highlight states whose spending is unusual compared to others.
df['state_id'] = df[STATE_COLUMN].astype('category').cat.codes

print(f"Data loaded and cleaned. Total rows: {len(df)}")
print(f"Target Column: {TARGET_COLUMN}")

Data loaded and cleaned. Total rows: 50
Target Column: score


In [3]:
# Define Target and Feature
X = df[['state_id']] # Feature: State ID
y = df[TARGET_COLUMN] # Target: MSPB Score (e.g., 1.00)

# 1. Train the model (Linear Regression is simple and effective for this)
model = LinearRegression()
model.fit(X, y)

# 2. Predict the MSPB score for all states
df['predicted_score'] = model.predict(X)

# 3. Calculate the Prediction Error (The AMA's Call to Action)
# Positive Error: Actual Spending is Higher than predicted (Inefficiency/Over-utilization risk)
# Negative Error: Actual Spending is Lower than predicted (Efficiency or Under-utilization risk)
df['prediction_error'] = df[TARGET_COLUMN] - df['predicted_score']

# 4. Calculate performance
r2 = r2_score(y, df['predicted_score'])

print(f"\nModel R-squared: {r2:.4f}")
print("--- Top 5 Highest Spending Outliers (Potential Inefficiency) ---")
print(df.sort_values(by='prediction_error', ascending=False)[[STATE_COLUMN, TARGET_COLUMN, 'prediction_error']].head())

# 5. Export the final data file for Tableau
FINAL_CSV_PATH = 'final_ama_msbp_analysis.csv'
df.to_csv(FINAL_CSV_PATH, index=False)
print(f"\nSUCCESS: Final analysis data saved to {FINAL_CSV_PATH}")


Model R-squared: 0.0167
--- Top 5 Highest Spending Outliers (Potential Inefficiency) ---
   state  score  prediction_error
20    LA   1.08          0.102759
34    NJ   1.07          0.097265
36    NV   1.05          0.078016
47    TX   1.03          0.061771
46    TN   1.02          0.051396

SUCCESS: Final analysis data saved to final_ama_msbp_analysis.csv


In [9]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np # Already imported, but good practice to keep with the analysis tools

# Load the final data file created in the previous step
FILE_PATH = 'final_ama_msbp_analysis.csv'
df = pd.read_csv(FILE_PATH)

# Ensure the state names are uppercase (Plotly's standard)
df['state'] = df['state'].str.upper()

# Ensure Prediction Error is numeric
df['prediction_error'] = pd.to_numeric(df['prediction_error'], errors='coerce')


## 1. Interactive U.S. Choropleth Map (The Outlier Visual)

# Determine the min/max for the color scale
# We use the absolute max to ensure a balanced, diverging color scale centered at 0
max_abs_error = df['prediction_error'].abs().max()

fig_map = px.choropleth(
    df,
    locations='state',
    locationmode='USA-states',
    color='prediction_error',
    scope='usa',
    # RdBu (Red-Blue) is a classic diverging scale for error/variance
    color_continuous_scale='RdBu_r', # Adding '_r' reverses it: Red=High, Blue=Low
    color_continuous_midpoint=0,     # Crucial: Centers the color scale at zero error
    title='AMA Focus: States with Unexpected Medicare Spending (Prediction Error)',
    hover_name='state',
    hover_data={
        'prediction_error': ':.4f',  # Show error to 4 decimal places
        'score': ':.4f',             # Show Actual MSPB Score
        'state': False
    }
)

# Customize the layout for clear presentation
fig_map.update_layout(
    coloraxis_colorbar=dict(
        title="Prediction Error",
        tickvals=[-max_abs_error, 0, max_abs_error],
        ticktext=['Highest Under-Spend', 'As Predicted', 'Highest Over-Spend']
    ),
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig_map.show()


## 🗺️ Map: High-Level Problem Identification

The **map** is the most powerful entry point. It provides an immediate geographic overview of where systemic healthcare issues exist.  
By coloring states according to **Prediction Error (unexplained spending)**, the map reveals whether certain regions consistently overspend or underspend compared to expectations.  

- **Why it matters:** Leaders quickly see patterns across the U.S.  
- **Impact:** Dark blue or red states highlight priority areas for deeper investigation.  

**Narrative to AMA:**  
*"This map shows where systemic healthcare issues hide. The darker the color, the greater the inefficiency or under-utilization.  
These states are our first targets for administrative reform."*

In [8]:
## 2. Top 10 Action List (Bar Chart)

# Filter for the top 10 highest positive errors (highest waste/inefficiency)
top_10_outliers = df.sort_values(by='prediction_error', ascending=False).head(10).reset_index(drop=True)

fig_bar = px.bar(
    top_10_outliers,
    x='state',
    y='prediction_error',
    color='prediction_error',
    color_continuous_scale=px.colors.sequential.Bluyl,
    title='Top 10 States for AMA Administrative Burden Audit (Highest Positive Error)',
    text='prediction_error'
)

# Customize layout
fig_bar.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig_bar.update_layout(xaxis_title="State (AMA Audit Priority)", yaxis_title="Prediction Error (Unexplained Over-Spending)")
fig_bar.update_coloraxes(showscale=False)

fig_bar.show()

## 📋 Bar Chart: Turning Findings into Action

While the map answers *where* the problem exists, the **bar chart** translates those findings into a concrete action plan.  
It ranks the **Top 10 states with the highest positive errors**, meaning they spend significantly more than predicted.  

- **Why it matters:** Provides a clear, ranked priority list.  
- **Impact:** AMA operational teams can target specific states (e.g., LA, NJ, NV) for administrative audits or lobbying campaigns.  

**Narrative to AMA:**  
*"This chart is our action list. These ten states represent the greatest inefficiencies.  
Focusing AMA resources here will yield the largest cost reductions and efficiency gains."*


In [7]:
import plotly.express as px

# Filter for just the states with the largest ABSOLUTE errors (top 10 highest positive
# and top 10 lowest negative errors) to highlight the critical outliers.
df['abs_error'] = df['prediction_error'].abs()
top_20_outliers = df.nlargest(20, 'abs_error')


# Create the Scatter Plot
fig_scatter = px.scatter(
    df,
    x='state_id',                     # X-axis: Simple predictor (State ID)
    y='score',                        # Y-axis: Actual MSPB Score (The Target)
    color='prediction_error',         # Color the points by the error magnitude
    color_continuous_scale='RdBu_r',  # Diverging color scale centered at 0
    color_continuous_midpoint=0,
    size='abs_error',                 # Size the points by the magnitude of the error
    hover_name='state',
    title='Model Failure is Policy Success: Identifying Extreme Spending Outliers',
    labels={'state_id': 'State ID (Model Input)', 'score': 'Actual MSPB Score (Spending Index)'}
)

# Add the 'Predicted Score' as a straight line (the model's expectation)
# This visually shows where states *should* land if the model were perfect.
fig_scatter.add_trace(
    px.line(df.sort_values(by='state_id'), x='state_id', y='predicted_score')
    .update_traces(line=dict(color='black', width=2, dash='dash'), name='Model Prediction')
    .data[0]
)

# Highlight the largest outliers with annotations
for index, row in top_20_outliers.iterrows():
    fig_scatter.add_annotation(
        x=row['state_id'],
        y=row['score'],
        text=row['state'],
        showarrow=True,
        arrowhead=1,
        ax=row['prediction_error'] * 500, # Move annotation away from the line
        ay=-50,
        font=dict(size=10, color='black'),
        bgcolor='rgba(255, 255, 255, 0.7)'
    )

fig_scatter.show()

## 🔬 Scatter Plot: Validating the Methodology

The **scatter plot** addresses the technical audience. It visualizes actual spending (Y-axis) against state IDs (X-axis), with a **predicted regression line** drawn across.  
The points furthest from the prediction line represent the **true outliers** that justify policy intervention.  

- **Why it matters:** Demonstrates that the analysis isn’t based on raw spending alone.  
- **Impact:** Confirms that outliers are systemic anomalies, not just high-cost states.  

**Narrative to AMA:**  
*"Our approach isolates structural inefficiencies. The relatively low R² shows geography alone doesn’t explain costs.  
Instead, we focus on states furthest from the predicted line—these are the anomalies that demand policy solutions."*


# 📢 AMA Policy Recommendation
This project recommends a data-first approach to AMA advocacy:

1. Administrative Audit: The AMA should allocate resources to investigate the Top 5 Positive Outlier states (LA, NJ, NV, TX, TN). This unexplained, excess spending is a strong signal of administrative waste and inefficient documentation/billing practices that contribute directly to physician burden.

2. Payment Reform Advocacy: Use the negative outlier states (those with spending much lower than predicted) to advocate for balanced payment reform. The AMA can either champion them as models of efficiency or investigate them for barriers to care caused by low reimbursement, aligning with the mission to support fair payment for physicians.

# ⚙️ Analytical Challenges, Impact, and Future Mitigation

## 1. Model Simplicity

* **Description & Impact:** Used **State ID** ($\text{X-axis}$) to predict the score ($\text{Y-axis}$). This simple linear model is weak ($\mathbf{R^2 \approx 0.0167}$), which is analytically correct for the narrative but lacks explanatory power.

* **How to Overcome (Future Steps):**
    * **Use a Multivariate Model:** Incorporate additional relevant features, such as state-level **population density, poverty rate, physician density** (from the old HRSA data idea), or **average patient age** (from other CMS files) to build a more robust, explainable model.

## 2. Handling Outliers

* **Description & Impact:** The Linear Regression model is sensitive to the extreme outliers (like LA and NJ). These states heavily influence the prediction line, potentially skewing the results for average-spending states.

* **How to Overcome (Future Steps):**
    * **Use Robust Regression:** Switch to an algorithm like **Random Forest Regressor** or **Ridge/Lasso Regression**. These models are less sensitive to extreme outliers and would likely provide a higher, more stable $\text{R}^2$ value, offering a more precise estimate of the expected spending score.

## 3. Data Aggregation (The "Black Box" Problem)

* **Description & Impact:** The MSPB data is pre-aggregated by CMS (state-level). This makes it easy but hides critical information about **specialty** (e.g., Are surgeons or primary care docs driving the cost?) and **procedure code** (e.g., Which specific service is over-utilized?).

* **How to Overcome (Future Steps):**
    * **Integrate Detailed Data (The AMA's Goal):** Link this MSPB summary data with the **raw CMS Physician and Other Supplier PUF** (the larger file initially searched for). This would identify which **specialties** and **procedure codes** are driving the $\text{prediction\_error}$ in Louisiana.